# Autonomous Trailer Agent
**Team:** Eric, Victor, Tashome

## 1. Introduction and agent overview

This notebook builds a knowledge-based agent that plans routes for an **autonomous trailer**. Given a start, a destination and the current state of the roads, the agent decides which roads it may use, finds the best route, checks that the trip can be completed on the fuel available and explains every decision it makes.

It builds on Exercise 3, where we searched a road graph for the shortest route. Search alone does not know that a road is flooded, that a bridge is too low for the trailer or that the tank will run dry halfway. This exercise adds a **knowledge base** and an **inference engine** on top of search to handle that.

### Agent description (PEAS)

| | |
|---|---|
| **Performance** | reach the destination by the shortest safe route, never use a blocked or unsuitable road, never run out of fuel, explain its choices |
| **Environment** | a road network of towns and roads with distances, plus changing road conditions (closures, floods, accidents, traffic, roadworks) and fuel stations |
| **Actuators** | outputs a route, its distance, any refuel stop and the reasons |
| **Sensors** | the knowledge base: the map, road condition data and the trailer's own state (fuel range, height) |

### How the parts fit

```
 user input  ->  interface (section 4)
                     |
                     v
 knowledge base (section 2)  ->  facts  ->  inference engine (section 3)  ->  search  ->  route + reasons
```

1. The **knowledge base** turns the map and the road condition data into facts.
2. The **inference engine** applies IF/THEN rules to those facts by forward chaining, filters the graph, calls search and checks fuel.
3. The **interface** asks for the trip details, calls `plan_route()` and prints the route, distance and reasons.

## 2. Knowledge base

The knowledge base stores everything the agent knows as facts: tuples in a set. The engine in section 3 only reads facts, so any data source can be used as long as it is turned into these formats:

| Fact | Meaning |
|---|---|
| `("edge", a, b, km)` | a two-way road between towns a and b, km long |
| `("closed", a, b)` | road is closed |
| `("accident", a, b)` | accident on the road |
| `("condition", a, b, "flooded" / "roadworks")` | road condition |
| `("traffic", a, b, "heavy")` | heavy traffic |
| `("max_height", a, b, m)` | height limit on the road (bridge, underpass) in metres |
| `("fuel_station", town)` | a fuel station in that town |
| `("fuel_range", km)` | how far the trailer can drive on a full tank |
| `("trailer_height", m)` | height of the trailer in metres |

### 2.1 Road network

The map is kept in the same form as Exercise 3: an adjacency dictionary of towns and road distances.

> **Note:** the graph below is a placeholder so that the notebook runs end to end. Replace `ROAD_GRAPH` with the Exercise 3 graph and distances. Nothing else needs to change.

In [1]:
# Placeholder road network (replace with the Exercise 3 graph)
ROAD_GRAPH = {
    "A": {"B": 75, "C": 140, "D": 118},
    "B": {"A": 75, "E": 71},
    "C": {"A": 140, "E": 151, "F": 99, "G": 80},
    "D": {"A": 118, "H": 111},
    "E": {"B": 71, "C": 151},
    "F": {"C": 99, "I": 211},
    "G": {"C": 80, "H": 146, "I": 97},
    "H": {"D": 111, "G": 146},
    "I": {"F": 211, "G": 97},
}

FUEL_STATIONS = ["C", "F", "G", "H"]

TRAILER = {"fuel_range": 500, "trailer_height": 4.2}

### 2.2 Road conditions

Road conditions change all the time, so they are kept apart from the map. `get_road_conditions()` is the single place they come from. It returns a list of condition records, which are then turned into facts. At the moment it returns sample data. It can be pointed at a traffic or road-status API, or at permitted scraped data, without changing the rest of the agent.

In [2]:
def get_road_conditions():
    """Return current road conditions as a list of dicts.
    Sample data for now; swap for an API or scraper that returns the same shape."""
    return [
        {"from": "C", "to": "G", "type": "traffic", "value": "heavy"},
        {"from": "D", "to": "H", "type": "max_height", "value": 4.0},
    ]

### 2.3 Turning data into facts

`build_kb()` combines the map, the fuel stations, the trailer's details and the road conditions into one fact set. Each road is stored once. The engine treats roads as two-way.

In [3]:
def build_kb(graph=None, stations=None, trailer=None, conditions=None):
    graph = ROAD_GRAPH if graph is None else graph
    stations = FUEL_STATIONS if stations is None else stations
    trailer = TRAILER if trailer is None else trailer
    conditions = get_road_conditions() if conditions is None else conditions

    facts = set()
    for a, neighbours in graph.items():
        for b, km in neighbours.items():
            if a < b:                                   # store each road once
                facts.add(("edge", a, b, km))
    facts |= {("fuel_station", s) for s in stations}
    facts.add(("fuel_range", trailer["fuel_range"]))
    facts.add(("trailer_height", trailer["trailer_height"]))

    for c in conditions:
        a, b, t, v = c["from"], c["to"], c["type"], c["value"]
        if t in ("closed", "accident"):
            facts.add((t, a, b))
        else:                                           # condition, traffic, max_height
            facts.add((t, a, b, v))
    return facts


KB = build_kb()
print(len(KB), "facts in the knowledge base, for example:")
for f in sorted(KB, key=str)[:8]:
    print("  ", f)

19 facts in the knowledge base, for example:
   ('edge', 'A', 'B', 75)
   ('edge', 'A', 'C', 140)
   ('edge', 'A', 'D', 118)
   ('edge', 'B', 'E', 71)
   ('edge', 'C', 'E', 151)
   ('edge', 'C', 'F', 99)
   ('edge', 'C', 'G', 80)
   ('edge', 'D', 'H', 111)


## 3. Inference engine

The inference engine decides **which roads the trailer may use, how costly each one is, and whether the trip can be completed on the fuel available**. It works on the facts produced by the knowledge base (section 2) and hands a filtered graph to the search algorithm from Exercise 3.

### 3.1 Representation

- **Facts** are Python tuples stored in a `set`, so a fact is either known or not and duplicates are impossible. The first element is the predicate name, for example `("edge", "Depot", "X", 40)` or `("closed", "Depot", "X")`.
- **Rules** are kept in a list. Each rule has a name, a plain-English IF/THEN statement (used in the reason trace), and a small function that reads the fact set and returns the facts it concludes.
- **Derived facts** use their own predicates (`avoid`, `penalty`, `needs_refuel`, `refuel_at`, `no_valid_route`) so it is always clear which facts came from the knowledge base and which were inferred.

### 3.2 Forward chaining

We chose **forward chaining** (data-driven reasoning) because the agent starts from what it knows about the roads and wants to find out everything that follows: which roads to avoid, which to penalise and whether to refuel. There is no single goal to prove, so backward chaining would not fit as well.

The loop is simple:

1. Go through every rule and ask it which facts it can conclude from the current fact set.
2. Remove anything already known. If something new is left, add it to the set and write a line in the reason trace.
3. Repeat until a full pass adds nothing new (a fixed point).

The loop always stops, because rules only add facts and the number of facts that can be built from the knowledge base is finite.

### 3.3 Link to search

Planning runs in two reasoning stages with search in between:

1. **Stage 1 (before search):** the rules work out `avoid` and `penalty` facts from closures, flooding, accidents, height limits, traffic and roadworks.
2. **Search:** `avoid` edges are removed from the graph and `penalty` edges have their cost multiplied. The search then finds the cheapest route. The distance reported back is always the real distance in km, not the penalised cost.
3. **Stage 2 (after search):** the route is added as facts (`route_km`, `on_route`) and the engine runs again to check fuel. It either accepts the route, adds a refuel stop or rejects the plan.

The search is written as a separate function (`shortest_path`). At the moment it is a Dijkstra stand-in so the engine could be tested on its own. At integration it is replaced by the Exercise 3 search without changing any rules.

In [4]:
"""Inference engine for the autonomous trailer agent (Victor's section).

Facts   : tuples in a set, e.g. ("edge", "A", "B", 40), ("blocked", "A", "B")
Rules   : list of Rule objects; each has a plain IF/THEN text and a function
          that looks at the fact set and returns the facts it concludes.
Chaining: forward chaining, repeated until no rule adds a new fact.
Link to search: stage 1 derives avoid/penalty facts -> filtered graph ->
          search -> stage 2 derives fuel facts from the found route.
"""
import heapq
from dataclasses import dataclass
from typing import Callable


@dataclass
class Rule:
    name: str
    text: str                      # plain IF/THEN, shown in the write-up
    fn: Callable[[set], set]       # facts -> set of concluded facts


def _undirected(a, b):
    return {(a, b), (b, a)}


# ---------------------------------------------------------------- rules ----
def r1(f):  # closed road
    return {("avoid", x, y) for k in f if k[0] == "closed" for x, y in _undirected(k[1], k[2])}

def r2(f):  # flooding
    return {("avoid", x, y) for k in f if k[0] == "condition" and k[3] == "flooded"
            for x, y in _undirected(k[1], k[2])}

def r3(f):  # accident
    return {("avoid", x, y) for k in f if k[0] == "accident" for x, y in _undirected(k[1], k[2])}

def r4(f):  # trailer too tall
    h = next((k[1] for k in f if k[0] == "trailer_height"), None)
    if h is None:
        return set()
    return {("avoid", x, y) for k in f if k[0] == "max_height" and k[3] < h
            for x, y in _undirected(k[1], k[2])}

def r5(f):  # heavy traffic
    return {("penalty", x, y, 1.5) for k in f if k[0] == "traffic" and k[3] == "heavy"
            for x, y in _undirected(k[1], k[2])}

def r6(f):  # roadworks
    return {("penalty", x, y, 1.3) for k in f if k[0] == "condition" and k[3] == "roadworks"
            for x, y in _undirected(k[1], k[2])}

def r7(f):  # no path found by search
    return {("no_valid_route", "no path on the filtered graph")} if ("search_failed",) in f else set()

def r8(f):  # route longer than range
    rng = next((k[1] for k in f if k[0] == "fuel_range"), None)
    km = next((k[1] for k in f if k[0] == "route_km"), None)
    return {("needs_refuel",)} if rng is not None and km is not None and km > rng else set()

def _cum(f):
    return {k[1]: k[2] for k in f if k[0] == "on_route"}

def r9(f):  # refuel station on route that splits the trip into two reachable legs
    if ("needs_refuel",) not in f:
        return set()
    rng = next(k[1] for k in f if k[0] == "fuel_range")
    total = next(k[1] for k in f if k[0] == "route_km")
    cum = _cum(f)
    ok = [(c, n) for n, c in cum.items() if ("fuel_station", n) in f and c <= rng and total - c <= rng]
    return {("refuel_at", min(ok)[1])} if ok else set()

def r10(f):  # needs fuel but no usable station
    if ("needs_refuel",) in f and not any(k[0] == "refuel_at" for k in f):
        return {("no_valid_route", "route exceeds fuel range and no reachable fuel stop")}
    return set()

RULES = [
    Rule("R1", "IF road is closed THEN avoid it", r1),
    Rule("R2", "IF road is flooded THEN avoid it", r2),
    Rule("R3", "IF road has an accident THEN avoid it", r3),
    Rule("R4", "IF road max height < trailer height THEN avoid it", r4),
    Rule("R5", "IF traffic is heavy THEN penalise the road (x1.5)", r5),
    Rule("R6", "IF road has roadworks THEN penalise the road (x1.3)", r6),
    Rule("R7", "IF search found no path THEN no valid route", r7),
    Rule("R8", "IF route km > fuel range THEN trailer needs a refuel", r8),
    Rule("R9", "IF refuel needed AND a station on the route leaves both legs within range THEN refuel there", r9),
    Rule("R10", "IF refuel needed AND no such station THEN no valid route", r10),
]


# ------------------------------------------------------- forward chaining ----
def forward_chain(facts, rules=RULES, trace=None):
    """Fire rules until nothing new is derived. Returns the extended fact set."""
    facts = set(facts)
    trace = trace if trace is not None else []
    changed = True
    while changed:
        changed = False
        for rule in rules:
            new = rule.fn(facts) - facts
            if new:
                facts |= new
                changed = True
                trace.append(f"{rule.name}: {rule.text}  ->  {sorted(map(str, new))}")
    return facts


# ----------------------------------------------------------------- search ----
def shortest_path(facts, start, dest):
    """Dijkstra over the graph with avoided edges removed and penalties applied.
    Stand-in for the Practical/Exercise 3 search; swap in the real one at integration."""
    avoid = {(k[1], k[2]) for k in facts if k[0] == "avoid"}
    pen = {(k[1], k[2]): k[3] for k in facts if k[0] == "penalty"}
    adj = {}
    for k in facts:
        if k[0] == "edge":
            _, a, b, km = k
            for x, y in _undirected(a, b):
                if (x, y) not in avoid:
                    adj.setdefault(x, []).append((y, km))
    pq, best = [(0.0, start, [start])], {}
    while pq:
        cost, node, path = heapq.heappop(pq)
        if node == dest:
            return path
        if best.get(node, float("inf")) <= cost:
            continue
        best[node] = cost
        for nxt, km in adj.get(node, []):
            heapq.heappush(pq, (cost + km * pen.get((node, nxt), 1.0), nxt, path + [nxt]))
    return None


def path_km(facts, path):
    d = {}
    for k in facts:
        if k[0] == "edge":
            d[(k[1], k[2])] = d[(k[2], k[1])] = k[3]
    legs = [d[(a, b)] for a, b in zip(path, path[1:])]
    return legs


# ------------------------------------------------------------ plan_route ----
def plan_route(facts, start, dest):
    """Returns dict: route (list|None), distance_km, refuel_at, reasons (list[str])."""
    trace = []
    f = forward_chain(set(facts) | {("start", start), ("dest", dest)}, trace=trace)   # stage 1
    path = shortest_path(f, start, dest)
    if path is None:
        f = forward_chain(f | {("search_failed",)}, trace=trace)
        return {"route": None, "distance_km": None, "refuel_at": None, "reasons": trace}

    legs = path_km(f, path)
    cum, run = [0], 0
    for l in legs:
        run += l
        cum.append(run)
    f |= {("route_km", run)} | {("on_route", n, c) for n, c in zip(path, cum)}
    trace.append(f"SEARCH: route {' -> '.join(path)}, {run} km")
    f = forward_chain(f, trace=trace)                                                 # stage 2

    bad = next((k for k in f if k[0] == "no_valid_route"), None)
    if bad:
        return {"route": None, "distance_km": None, "refuel_at": None, "reasons": trace}
    stop = next((k[1] for k in f if k[0] == "refuel_at"), None)
    return {"route": path, "distance_km": run, "refuel_at": stop, "reasons": trace}


### 3.4 Rules

| Rule | IF | THEN | Why |
|---|---|---|---|
| R1 | road is closed | avoid the road | cannot be driven |
| R2 | road is flooded | avoid the road | unsafe for a loaded trailer |
| R3 | there is an accident on the road | avoid the road | blocked or very slow |
| R4 | road's height limit < trailer height | avoid the road | trailer will not fit under the bridge |
| R5 | traffic on the road is heavy | penalise it (cost x1.5) | still usable, but only if it is clearly better |
| R6 | road has roadworks | penalise it (cost x1.3) | slower, but still open |
| R7 | search found no path | no valid route | nothing left after filtering |
| R8 | route length > fuel range | trailer needs to refuel | cannot finish on the current tank |
| R9 | refuel needed AND a station on the route leaves both legs within range | refuel at that station (the earliest one) | splits the trip into two legs the trailer can drive |
| R10 | refuel needed AND no such station exists | no valid route | trip is impossible with this fuel |

Roads are two-way, so rules R1 to R6 apply in both directions. Rules R7 to R10 only fire in stage 2, because they depend on facts that exist only after search.

The cell below prints the rules straight from the code, so the table and the program cannot drift apart.

In [5]:
for r in RULES:
    print(r.name, '-', r.text)

R1 - IF road is closed THEN avoid it
R2 - IF road is flooded THEN avoid it
R3 - IF road has an accident THEN avoid it
R4 - IF road max height < trailer height THEN avoid it
R5 - IF traffic is heavy THEN penalise the road (x1.5)
R6 - IF road has roadworks THEN penalise the road (x1.3)
R7 - IF search found no path THEN no valid route
R8 - IF route km > fuel range THEN trailer needs a refuel
R9 - IF refuel needed AND a station on the route leaves both legs within range THEN refuel there
R10 - IF refuel needed AND no such station THEN no valid route


### 3.5 Testing

The expected answer for each scenario was written down **before** the code was run. The test graph is a small hand-made one so that the right answer can be checked by hand:

- Depot to X to Port = 40 + 40 = **80 km** (shortest)
- Depot to Y to Port = 50 + 50 = **100 km**
- Fuel stations at X and Y

| # | Scenario | Expected result |
|---|---|---|
| 1 | Nothing blocked, range 200 km | Depot to X to Port, 80 km, no stop |
| 2 | Depot to X closed | reroute via Y, 100 km |
| 3 | Range 70 km | Depot to X to Port, refuel at X (40 km + 40 km) |
| 4 | Depot to X closed and range 70 km | via Y, refuel at Y (50 km + 50 km) |
| 5 | Depot to X closed and accident on Depot to Y | no valid route |

In [6]:
"""Phase 4 scenarios. Expected answers were written before running."""

# Hand-written KB (placeholder until Eric's real graph is plugged in)
BASE = {
    ("edge", "Depot", "X", 40), ("edge", "X", "Port", 40),   # 80 km
    ("edge", "Depot", "Y", 50), ("edge", "Y", "Port", 50),   # 100 km
    ("fuel_station", "X"), ("fuel_station", "Y"),
    ("fuel_range", 200),
}

def run(extra=(), remove=()):
    kb = (BASE - set(remove)) | set(extra)
    return plan_route(kb, "Depot", "Port")

def test_1_nothing_blocked():
    r = run()
    assert r["route"] == ["Depot", "X", "Port"] and r["distance_km"] == 80 and r["refuel_at"] is None

def test_2_shortest_blocked_reroute():
    r = run({("closed", "Depot", "X")})
    assert r["route"] == ["Depot", "Y", "Port"] and r["distance_km"] == 100

def test_3_low_fuel_adds_stop():
    r = run({("fuel_range", 70)}, remove={("fuel_range", 200)})
    assert r["route"] == ["Depot", "X", "Port"] and r["refuel_at"] == "X"

def test_4_blocked_and_low_fuel():
    r = run({("closed", "Depot", "X"), ("fuel_range", 70)}, remove={("fuel_range", 200)})
    assert r["route"] == ["Depot", "Y", "Port"] and r["refuel_at"] == "Y"

def test_5_everything_blocked():
    r = run({("closed", "Depot", "X"), ("accident", "Depot", "Y")})
    assert r["route"] is None



In [7]:
for n, fn in list(globals().items()):
    if n.startswith('test_'):
        fn(); print('PASS', n)

PASS test_1_nothing_blocked
PASS test_2_shortest_blocked_reroute
PASS test_3_low_fuel_adds_stop
PASS test_4_blocked_and_low_fuel
PASS test_5_everything_blocked


### 3.6 Worked reason trace

Scenario 4 (blocked road and low fuel) uses both stages, so it is the most useful one to follow:

1. **Stage 1:** the knowledge base says `("closed", "Depot", "X")`. **R1** fires and adds `avoid` for Depot to X in both directions. No other stage 1 rule has anything to work on, so chaining stops.
2. **Search:** with Depot to X removed, the only route is Depot to Y to Port, 100 km.
3. **Stage 2:** `route_km = 100` and `fuel_range = 70`, so **R8** fires and adds `needs_refuel`.
4. On the next pass **R9** looks at stations on the route. Y is 50 km in, which is within 70 km, and leaves 50 km to go, also within 70 km. So it adds `refuel_at Y`.
5. **R10** does not fire because a stop was found. Nothing new is added, so chaining stops and the plan is accepted.

The output below is the actual trace printed by the program for this scenario.

In [8]:
r = run({('closed','Depot','X'), ('fuel_range',70)}, remove={('fuel_range',200)})
print(r['route'], r['distance_km'], 'refuel at', r['refuel_at'])
for line in r['reasons']:
    print(line)

['Depot', 'Y', 'Port'] 100 refuel at Y
R1: IF road is closed THEN avoid it  ->  ["('avoid', 'Depot', 'X')", "('avoid', 'X', 'Depot')"]
SEARCH: route Depot -> Y -> Port, 100 km
R8: IF route km > fuel range THEN trailer needs a refuel  ->  ["('needs_refuel',)"]
R9: IF refuel needed AND a station on the route leaves both legs within range THEN refuel there  ->  ["('refuel_at', 'Y')"]


### 3.7 Limitations

- Penalty values (x1.5, x1.3) are fixed guesses. Live traffic data from the knowledge base could set them instead.
- The fuel rule only plans **one** stop. A very long route that needs two stops is reported as having no valid route.
- The trailer is assumed to refuel to a full tank.

## 4. Interface and sample runs

The interface is the only part the user deals with. It collects the start, the destination and any changes to the road conditions or the fuel range, builds the knowledge base, calls `plan_route()` and prints the result in plain language.

In [9]:
def show_plan(result, start, dest):
    print(f"Trip: {start} -> {dest}")
    if result["route"] is None:
        print("Result: NO VALID ROUTE")
    else:
        print("Route: ", " -> ".join(result["route"]))
        print("Distance:", result["distance_km"], "km")
        print("Refuel at:", result["refuel_at"] or "not needed")
    print("Reasons:")
    for line in result["reasons"]:
        print("  -", line)
    print()


def trip(start, dest, extra_conditions=(), fuel_range=None):
    """Plan one trip. extra_conditions are added on top of get_road_conditions()."""
    trailer = dict(TRAILER)
    if fuel_range is not None:
        trailer["fuel_range"] = fuel_range
    kb = build_kb(trailer=trailer, conditions=get_road_conditions() + list(extra_conditions))
    result = plan_route(kb, start, dest)
    show_plan(result, start, dest)
    return result

### 4.1 Asking the user

Set `INTERACTIVE = True` to type a trip in by hand. It is off by default so that the notebook can run top to bottom without stopping.

In [10]:
INTERACTIVE = False

if INTERACTIVE:
    towns = sorted(ROAD_GRAPH)
    print("Towns:", ", ".join(towns))
    start = input("Start: ").strip().upper()
    dest = input("Destination: ").strip().upper()
    rng = input(f"Fuel range in km [{TRAILER['fuel_range']}]: ").strip()
    blocked = input("Closed roads, e.g. A-B,C-G (blank for none): ").strip()
    extra = [{"from": p.split("-")[0].strip().upper(), "to": p.split("-")[1].strip().upper(),
              "type": "closed", "value": True}
             for p in blocked.split(",") if "-" in p]
    if start not in ROAD_GRAPH or dest not in ROAD_GRAPH:
        print("Unknown town.")
    else:
        trip(start, dest, extra, int(rng) if rng else None)

### 4.2 Sample runs on the road network

The same five situations as the engine tests (section 3.5), this time on the full knowledge base from section 2. All trips go from **A to I**.

In [11]:
print("1) Normal conditions")
r1 = trip("A", "I")

print("2) Shortest route blocked")
r2 = trip("A", "I", [{"from": "C", "to": "G", "type": "closed", "value": True}])

print("3) Low fuel")
r3 = trip("A", "I", fuel_range=250)

print("4) Blocked road and low fuel")
r4 = trip("A", "I", [{"from": "C", "to": "G", "type": "closed", "value": True}], fuel_range=250)

print("5) Everything blocked")
r5 = trip("A", "I", [{"from": "C", "to": "G", "type": "closed", "value": True},
                     {"from": "F", "to": "I", "type": "accident", "value": True}])

1) Normal conditions
Trip: A -> I
Route:  A -> C -> G -> I
Distance: 317 km
Refuel at: not needed
Reasons:
  - R4: IF road max height < trailer height THEN avoid it  ->  ["('avoid', 'D', 'H')", "('avoid', 'H', 'D')"]
  - R5: IF traffic is heavy THEN penalise the road (x1.5)  ->  ["('penalty', 'C', 'G', 1.5)", "('penalty', 'G', 'C', 1.5)"]
  - SEARCH: route A -> C -> G -> I, 317 km

2) Shortest route blocked
Trip: A -> I
Route:  A -> C -> F -> I
Distance: 450 km
Refuel at: not needed
Reasons:
  - R1: IF road is closed THEN avoid it  ->  ["('avoid', 'C', 'G')", "('avoid', 'G', 'C')"]
  - R4: IF road max height < trailer height THEN avoid it  ->  ["('avoid', 'D', 'H')", "('avoid', 'H', 'D')"]
  - R5: IF traffic is heavy THEN penalise the road (x1.5)  ->  ["('penalty', 'C', 'G', 1.5)", "('penalty', 'G', 'C', 1.5)"]
  - SEARCH: route A -> C -> F -> I, 450 km

3) Low fuel
Trip: A -> I
Route:  A -> C -> G -> I
Distance: 317 km
Refuel at: C
Reasons:
  - R4: IF road max height < trailer height 

### 4.3 Results

| # | Situation | Result |
|---|---|---|
| 1 | Normal conditions | A -> C -> G -> I, 317 km, no stop. D-H avoided (too low), C-G taken despite traffic because it is still cheaper |
| 2 | Shortest route blocked | C-G closed, rerouted A -> C -> F -> I, 450 km |
| 3 | Low fuel | range 250 km < 317 km, refuel at C (140 km + 177 km) |
| 4 | Blocked road and low fuel | rerouted via F, 450 km, refuel at F (239 km + 211 km) |
| 5 | Everything blocked | C-G closed and accident on F-I, no valid route |

In every case the agent avoided the roads it was told about and gave the rule that made it do so. The height limit on D-H (4.0 m, trailer 4.2 m) is avoided in every run by rule R4, even though nobody asked for it, which shows the knowledge base and rules working together rather than the user having to spot problems.

## 5. Conclusion

Adding a knowledge base and rules to the Exercise 3 search turns a shortest-path finder into an agent that can deal with real road problems and explain itself. The rules are easy to read and change, and keeping the facts, the rules and the search apart means each part could be built and tested on its own.